# S02 - Credit and term spreads (macro / credit)

**Hypothesis.** Widening corporate credit spreads signal rising default risk and tighter
financing conditions, which tend to precede equity drawdowns; an inverted yield curve
predicts recessions. Holding T-bills while credit spreads widen (or the curve is inverted)
should avoid part of the bear markets that SMA200 catches late.

| Rule | Risk-on (hold S&P 500) when ... |
|---|---|
| credit_ma6 / credit_ma12 | BAA - AAA spread <= its 6 / 12-month moving average |
| term_pos | AAA - TB3MS > 0 (curve not inverted) |
| credit12_and_term | both of the above (k = 12) |

**Inputs and point-in-time lags**

| Series | Source | Start | Lag used |
|---|---|---|---|
| BAA, AAA Moody's corporate yields, monthly averages | FRED `BAA`, `AAA` | 1919 | month M usable from M+1 + 7 days |
| 3-month T-bill, monthly average | FRED `TB3MS` | 1934 | month M usable from M+1 + 7 days |
| S&P 500 total return, T-bills | `bt_long.long_sample` | 1934 | daily, decide at close, next-close fill |

Long rate = Moody's AAA (not Shiller's pre-1953 long rate, which is interpolated from annual
data). Monthly check, next-close execution, 5 bp per switch: identical to the SMA200 benchmark.

**Sources.** Fama & French (1989), *Business conditions and expected returns*,
doi:10.1016/0304-405X(89)90095-0; Estrella & Mishkin (1996), *The yield curve as a predictor
of U.S. recessions*, NBER w5379; Gilchrist & Zakrajsek (2012), *Credit spreads and business
cycle fluctuations*, AER doi:10.1257/aer.102.4.1692. Data: https://fred.stlouisfed.org
(FRED terms of use apply).

> Decision-support research only; never places orders.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
sys.path.insert(0, str(_root / "notebooks" / "backtest"))
import bt_common as bc  # noqa: E402
import bt_long  # noqa: E402
import bt_periods as bp  # noqa: E402
import bt_pit  # noqa: E402
import bt_registry as br  # noqa: E402
import bt_stats as bs  # noqa: E402
import seed_registry  # noqa: E402
import stage2 as s2  # noqa: E402

bc.setup_plotly("cdn")
pd.set_option("display.width", 180)
TRIAL_ID, STRATEGY, NOTEBOOK = 'S02', 'credit_term_spread', 'macro/credit_term_spread.ipynb'
PROTOCOL, REBALANCE = 'main', 'monthly'

sample = bt_long.long_sample()
px, cash = sample["px"], sample["cash"]
bench = bp.benchmark_returns(px, cash)  # SMA200 monthly / B&H / 60-40 / T-bills, frozen settings
print("prices", px.index[0].date(), "->", px.index[-1].date(), "| registry trials so far:", br.trial_count())

prices 1934-02-01 -> 2026-10-19 | registry trials so far: 5


## 1. Data and point-in-time inputs

In [2]:
baa, aaa, tb3 = (bt_long.fred_series(s) for s in ("BAA", "AAA", "TB3MS"))
inputs = {"px": px, "baa": baa, "aaa": aaa, "tb3": tb3}
print("BAA/AAA", baa.index[0].date(), "->", baa.index[-1].date(), "| TB3MS", tb3.index[0].date(), "->", tb3.index[-1].date())
display(pd.DataFrame({"credit (BAA-AAA)": baa - aaa, "term (AAA-TB3MS)": aaa - tb3}).tail(4))

BAA/AAA 1919-01-01 -> 2026-09-01 | TB3MS 1934-01-01 -> 2026-09-01


,credit (BAA-AAA),term (AAA-TB3MS)
observation_date,,
2026-06-01,0.48,1.86
2026-07-01,0.43,2.03
2026-08-01,0.44,2.16
2026-09-01,0.44,2.09


## 2. Signal and variant grid (fixed before any backtest)

In [3]:
LAG = dict(months=0, days=7)  # monthly average for M known early M+1; 7-day margin


def build_signal(inputs, rule="credit", k=12):
    credit = (inputs["baa"] - inputs["aaa"]).dropna()
    term = (inputs["aaa"] - inputs["tb3"]).dropna()
    ma = credit.rolling(k).mean()
    credit_on = (credit <= ma).astype(float).where(ma.notna())
    term_on = (term > 0).astype(float).where(term.notna())
    monthly = {"credit": credit_on, "term": term_on, "both": credit_on * term_on}[rule]
    return bt_pit.monthly_to_daily_pit(monthly, inputs["px"].index, **LAG)


VARIANTS = {
    "credit_ma6": dict(rule="credit", k=6),
    "credit_ma12": dict(rule="credit", k=12),
    "term_pos": dict(rule="term"),
    "credit12_and_term": dict(rule="both", k=12),
}
PARAMS_TEXT = {n: f"{n}; FRED BAA/AAA/TB3MS monthly, lag M+1+7d; monthly; next_close; 5bp" for n in VARIANTS}

In [4]:
credit = baa - aaa
fig = go.Figure()
fig.add_trace(go.Scatter(x=credit.index, y=credit, name="BAA - AAA (pp)"))
fig.add_trace(go.Scatter(x=credit.index, y=credit.rolling(12).mean(), name="12-month MA", line=dict(dash="dash")))
fig.add_trace(go.Scatter(x=aaa.index, y=aaa - tb3, name="AAA - TB3MS (pp)", yaxis="y2", opacity=0.6))
fig.update_layout(title="Credit and term spreads", height=420, legend=dict(orientation="h"),
                  yaxis2=dict(overlaying="y", side="right", title="term"))
fig.show()

## 3. Look-ahead checks

Every variant is recomputed with all inputs cut at past dates (monthly inputs: completed months only) and must equal the full-data signal on those dates (`stage2.truncation_test`). No backtest runs unless all pass.

In [5]:
N_CUTS = 30
trunc = {}
for name, params in VARIANTS.items():
    trunc[name] = s2.truncation_test(build_signal, inputs, n_cuts=N_CUTS, **params)
display(pd.DataFrame({k: v.row() for k, v in trunc.items()}).T)
bad = [k for k, v in trunc.items() if not v.passed]
assert not bad, f"look-ahead detected in variants {bad}: fix before any backtest"
print("all variants pass the truncation look-ahead test")

,cuts,mismatches,max_abs_diff,passed
credit_ma6,30,0,0.0,True
credit_ma12,30,0,0.0,True
term_pos,30,0,0.0,True
credit12_and_term,30,0,0.0,True


all variants pass the truncation look-ahead test


## 4. Development-period sweep

Each variant is scored on the **dev** period only and registered as a trial. The best dev Sharpe is chosen; validation and holdout results are never used for the choice.

In [6]:
signals = {name: build_signal(inputs, **params) for name, params in VARIANTS.items()}
sweep, CHOSEN = s2.dev_sweep(px, cash, bench, signals, PARAMS_TEXT, trial_id=TRIAL_ID, strategy=STRATEGY,
                             notebook=NOTEBOOK, protocol=PROTOCOL, rebalance=REBALANCE)
display(sweep)
print("chosen on dev Sharpe only:", CHOSEN, "|", PARAMS_TEXT[CHOSEN])

,dev_sharpe,d_sma200,d_bh,time_in_market,switches_pa,dev_start
credit_ma6,0.562897,-0.080838,0.051898,0.571667,2.441316,1935-02-08
credit_ma12,0.491852,-0.151884,-0.019147,0.598882,1.512158,1935-02-08
term_pos,0.570505,-0.07323,0.059506,0.952737,0.328102,1935-02-18
credit12_and_term,0.513992,-0.129744,0.002993,0.588427,1.512913,1935-02-18


chosen on dev Sharpe only: term_pos | term_pos; FRED BAA/AAA/TB3MS monthly, lag M+1+7d; monthly; next_close; 5bp


## 5. Frozen evaluation of the chosen variant

Frozen primary metric (`PLAN.md` section 2): beat SMA200 and buy & hold in dev **and** val, one-sided block-bootstrap p <= 0.10 vs SMA200 in both, deflated Sharpe >= 0.95 with the registry trial count. Holdout is confirmatory only.

In [7]:
signal = signals[CHOSEN]
runs, cmp, stats, res = s2.evaluate(px, cash, bench, signal, protocol=PROTOCOL, rebalance=REBALANCE)
display(cmp.round(3))
display(stats)
display(res.to_frame().round(3))
print(f"n_trials={res.n_trials} | DSR={res.dsr:.3f} | passes={res.passes} | holdout confirms={res.holdout_confirms}")
display(pd.Series(res.checks, name="check"))

,strategy,bh,d_bh,sma200,d_sma200,mix60,d_mix60
dev,0.571,0.511,0.060,0.644,-0.073,0.518,0.052
val,0.319,0.319,0.000,0.572,-0.253,0.309,0.010
holdout,0.698,0.781,-0.082,0.658,0.041,0.779,-0.081
full,0.530,0.513,0.017,0.630,-0.100,0.515,0.015


,CAGR,Vol,Sharpe,Sortino,Max DD,Calmar,Worst day,MaxDD peak,MaxDD recovered,Final x,Time in market,Switches,Switches / yr,Cost drag / yr
dev,0.120241,0.153419,0.570505,0.799586,-0.515132,0.233417,-0.204547,1937-03-10,1944-03-16,507.313999,0.952737,18,0.328102,0.000164
val,0.082095,0.186119,0.319049,0.453099,-0.552502,0.148587,-0.090259,2007-10-09,-,4.843046,1.0,0,0.0,0.0
holdout,0.124171,0.167083,0.698256,0.980877,-0.337905,0.367472,-0.119799,2020-02-19,2020-08-10,7.135043,0.909564,6,0.357387,0.000179
full,0.11249,0.163611,0.530006,0.745886,-0.552502,0.203602,-0.204547,2007-10-09,2012-04-02,17530.406658,0.955167,24,0.26182,0.000131


,Sharpe,d_sma200,d_bh,p_vs_sma200
dev,0.571,-0.073,0.060,0.831
val,0.319,-0.253,0.000,0.929
holdout,0.698,0.041,-0.082,NaN
full,0.530,-0.100,0.017,NaN


n_trials=9 | DSR=1.000 | passes=False | holdout confirms=True


dev_beats_sma200        False
dev_beats_bh             True
dev_p_le_0.1            False
val_beats_sma200        False
val_beats_bh            False
val_p_le_0.1            False
dsr_ge_0.95              True
holdout_d_sma200_pos     True
Name: check, dtype: bool

In [8]:
s2.plot_growth(runs, bench, f'{TRIAL_ID} {CHOSEN}', PROTOCOL).show()

## 6. Robustness: costs, placebo, peek test, decades

In [9]:
cs = bs.cost_sensitivity(px, signal, cash, rebalance=REBALANCE)
display(cs.round(3))
sma_full = bs.excess_sharpe(bench["sma200"].reindex(runs["full"].index), bench["tbill"].reindex(runs["full"].index))
be = bs.breakeven_cost_bps(px, signal, cash, bench_sharpe=sma_full, rebalance=REBALANCE)
print(f"break-even one-way cost vs SMA200 Sharpe over the same window ({sma_full:.3f}): {be:.1f} bp")
pl = bs.placebo_test(px, signal, cash, n_placebo=200, rebalance=REBALANCE, rng=np.random.default_rng(bs.SEED))
display(pd.Series(pl, name="placebo").to_frame())
peek = bt_pit.lookahead_test(px, signal, cash, rebalance=REBALANCE)
print(f"peek test: Sharpe by shift {peek.sharpe_by_shift.round(3).to_dict()} | gain {peek.peek_gain:+.3f}")
display(s2.decade_table(runs["full"], bench).style.format("{:.2f}"))

,Sharpe,CAGR,Max DD,turnover_pa
cost_bps,,,,
0,0.513,0.109,-0.553,0.27
5,0.512,0.109,-0.553,0.27
10,0.511,0.109,-0.553,0.27
25,0.508,0.109,-0.553,0.27
50,0.504,0.108,-0.553,0.27


break-even one-way cost vs SMA200 Sharpe over the same window (0.630): 0.0 bp


,placebo
stat,Sharpe
real,0.511754
placebo_mean,0.498667
placebo_q95,0.535307
quantile,0.495
p_one_sided,0.505
n_placebo,200
time_in_market,0.955408


peek test: Sharpe by shift {-21: 0.532, -5: 0.515, -1: 0.51, 0: 0.512, 1: 0.512, 5: 0.511, 21: 0.52} | gain +0.020


,Sharpe,Sharpe B&H,Sharpe SMA200,CAGR,CAGR B&H,CAGR SMA200,time in market
1930s,0.54,0.26,0.41,0.12,0.03,0.05,1.00
1940s,0.61,0.61,0.57,0.09,0.09,0.06,1.00
1950s,1.42,1.42,1.51,0.19,0.19,0.18,1.00
1960s,0.35,0.40,0.43,0.07,0.08,0.07,0.99
1970s,0.10,0.03,0.37,0.07,0.06,0.09,0.88
1980s,0.62,0.53,0.58,0.19,0.17,0.17,0.88
1990s,0.92,0.92,0.64,0.18,0.18,0.13,1.00
2000s,-0.05,-0.05,0.50,-0.01,-0.01,0.08,1.00
2010s,0.90,0.90,0.71,0.14,0.14,0.09,1.00
2020s,0.48,0.67,0.58,0.11,0.15,0.10,0.78


## 7. Register the result

In [10]:
note = ("passes" if res.passes else "fails") + f" frozen metric; chosen {CHOSEN} on dev; protocol {PROTOCOL}"
rows = s2.register_final(TRIAL_ID, STRATEGY, NOTEBOOK, PARAMS_TEXT[CHOSEN], res, note=note, header=seed_registry.HEADER)
display(rows)
print("registry trial count:", br.trial_count())

,trial_id,date,strategy,notebook,params,period,sharpe,d_sma200,d_bh,p_vs_sma200,dsr,n_trials,passes,status,note
28,S02-v1/dev,2026-10-20,credit_term_spread,macro/credit_term_spread.ipynb,"credit_ma6; FRED BAA/AAA/TB3MS monthly, lag M+...",dev,0.5629,-0.0808,0.0519,,,,,run,"dev-only sweep, variant credit_ma6"
29,S02-v2/dev,2026-10-20,credit_term_spread,macro/credit_term_spread.ipynb,"credit_ma12; FRED BAA/AAA/TB3MS monthly, lag M...",dev,0.4919,-0.1519,-0.0191,,,,,run,"dev-only sweep, variant credit_ma12"
30,S02-v3/dev,2026-10-20,credit_term_spread,macro/credit_term_spread.ipynb,"term_pos; FRED BAA/AAA/TB3MS monthly, lag M+1+...",dev,0.5705,-0.0732,0.0595,,,,,run,"dev-only sweep, variant term_pos"
31,S02-v4/dev,2026-10-20,credit_term_spread,macro/credit_term_spread.ipynb,"credit12_and_term; FRED BAA/AAA/TB3MS monthly,...",dev,0.5140,-0.1297,0.0030,,,,,run,"dev-only sweep, variant credit12_and_term"
32,S02/dev,2026-10-20,credit_term_spread,macro/credit_term_spread.ipynb,"term_pos; FRED BAA/AAA/TB3MS monthly, lag M+1+...",dev,0.5705,-0.0732,0.0595,0.8309,,9,False,fail,fails frozen metric; chosen term_pos on dev; p...
33,S02/val,2026-10-20,credit_term_spread,macro/credit_term_spread.ipynb,"term_pos; FRED BAA/AAA/TB3MS monthly, lag M+1+...",val,0.3190,-0.2532,0.0000,0.9287,,9,False,fail,fails frozen metric; chosen term_pos on dev; p...
34,S02/holdout,2026-10-20,credit_term_spread,macro/credit_term_spread.ipynb,"term_pos; FRED BAA/AAA/TB3MS monthly, lag M+1+...",holdout,0.6983,0.0407,-0.0825,,,9,False,fail,fails frozen metric; chosen term_pos on dev; p...
35,S02/full,2026-10-20,credit_term_spread,macro/credit_term_spread.ipynb,"term_pos; FRED BAA/AAA/TB3MS monthly, lag M+1+...",full,0.5300,-0.0997,0.0167,,0.9998,9,False,fail,fails frozen metric; chosen term_pos on dev; p...


registry trial count: 9


## 8. Current signal

In [11]:
last = signal.dropna()
print(f"latest signal ({CHOSEN}) on {last.index[-1].date()}: equity weight {last.iloc[-1]:.2f}")
chk = bc.check_days(px.index, REBALANCE)
display(last[chk.reindex(last.index).fillna(False)].tail(6).to_frame("equity weight at decision"))
print("Research output only -- not a recommendation or an order.")

latest signal (term_pos) on 2026-10-19: equity weight 1.00


,equity weight at decision
Date,
2026-04-30,1.0
2026-05-29,1.0
2026-06-30,1.0
2026-07-31,1.0
2026-08-31,1.0
2026-09-30,1.0


Research output only -- not a recommendation or an order.


## 9. Evidence summary (computed)

In [12]:
for line in s2.evidence_lines(res, trunc[CHOSEN], peek, pl, be):
    print("*", line)

* Truncation look-ahead test: PASSED on 30 cut dates; peek test gain +0.020.
* dev: Sharpe 0.571 (vs SMA200 -0.073, vs B&H +0.060, p vs SMA200 0.831)
* val: Sharpe 0.319 (vs SMA200 -0.253, vs B&H +0.000, p vs SMA200 0.929)
* holdout: Sharpe 0.698 (vs SMA200 +0.041, vs B&H -0.082)
* Deflated Sharpe 1.000 with n_trials=9.
* Placebo: real 0.512 vs placebo 95th pct 0.535 (p 0.505); break-even cost vs SMA200 full 0.0 bp.
* Verdict: FAILS the frozen primary metric (failed checks: dev_beats_sma200, dev_p_le_0.1, val_beats_sma200, val_beats_bh, val_p_le_0.1); holdout confirms.


## 10. Caveats

* Monthly averages: the signal reacts with up to ~5 weeks' delay (month average + release lag).
* AAA - TB3MS is a corporate-curve proxy for the Treasury term spread; it inverts less often.
* Moody's index composition has changed over time; FRED values are not revised in practice.
* Only four variants were tried, all registered; the best dev variant is evaluated once.

## 11. Interpretation

* The best development variant (`term_pos`, AAA - TB3MS > 0) is in equities ~95% of the time, so it is
  essentially buy & hold: Sharpe 0.57 / 0.32 in dev / val vs SMA200 0.64 / 0.57. It never beats SMA200
  in dev or val (bootstrap p 0.83 / 0.93) and ties B&H in val. **Fails** the frozen metric.
* The credit-spread rules (spread above its 6 or 12-month average = risk-off) switch more often but did
  worse in dev (0.49-0.56); with monthly averages and a release lag they react after the drawdown is
  under way, the same weakness as SMA200 without its robustness.
* The placebo confirms there is no timing information (real Sharpe at the 50th percentile of random
  regimes with the same time in market). The deflated Sharpe of 1.00 only says the Sharpe is above
  zero after 9 trials, not that it beats SMA200; the bootstrap vs SMA200 is the binding test.